In [3]:
# ---
# title: "Fase 1 - Adquisicion y preprocesamiento en GEE (AOI acotado)"
# author: "Lina Maria Quintero Fonseca"
# format: html
# ---

# Inicializacion robusta de Earth Engine via Application Default Credentials.
# Si las ADC no estan configuradas, ejecuta en la terminal del contenedor:
#   gcloud auth application-default login --no-launch-browser
#   gcloud auth application-default set-quota-project basic-buttress-338101

import ee
import geemap
import json
from google.auth import default

PROJECT_ID = "basic-buttress-338101"

creds, _ = default()
ee.Initialize(creds, project=PROJECT_ID)
print(f"GEE inicializado con proyecto: {PROJECT_ID}")

GEE inicializado con proyecto: basic-buttress-338101


In [6]:
# ----------------------------------------------------------------------------
# AOI acotado: SFF CGSM + Via Parque Isla de Salamanca (RUNAP)
# Generado por src/python/aoi_acotado.py - area ~835 km2
# ----------------------------------------------------------------------------
AOI_PATH = "../data/raw/cgsm_aoi_acotado_4326.geojson"
with open(AOI_PATH) as h:
    aoi_geojson = json.load(h)

# Tomar la geometria del primer feature (MultiPolygon SFF + VPI)
aoi_geom_dict = aoi_geojson["features"][0]["geometry"]
aoi = ee.Geometry(aoi_geom_dict)

area_km2 = aoi.area().divide(1e6).getInfo()
print(f"Area AOI acotado: {area_km2:,.1f} km2 (esperado ~835)")
print(f"Origen: {AOI_PATH}")

# Visualizar
Map = geemap.Map(center=[10.7, -74.5], zoom=10)
Map.addLayer(ee.Feature(aoi), {"color": "1B5E20"}, "CGSM AOI acotado")
Map

Area AOI acotado: 839.4 km2 (esperado ~835)
Origen: ../data/raw/cgsm_aoi_acotado_4326.geojson


Map(center=[10.7, -74.5], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright',…

In [7]:
# Celda eliminada. El AOI se carga desde geojson en la celda 1.
print("OK")

OK


In [8]:
# Función de máscara de nubes Sentinel-2
def mask_s2_clouds(image):
    qa = image.select('QA60')
    cloud_bit = 1 << 10
    cirrus_bit = 1 << 11
    mask = qa.bitwiseAnd(cloud_bit).eq(0).And(
           qa.bitwiseAnd(cirrus_bit).eq(0))
    return image.updateMask(mask)

# Función de índices espectrales
def add_indices(image):
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
    ndwi = image.normalizedDifference(['B3', 'B8']).rename('NDWI')
    cmri = ndvi.subtract(ndwi).rename('CMRI')
    return image.addBands([ndvi, ndwi, cmri])

# Construir colección Sentinel-2 (2017-2025)
s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
      .filterBounds(aoi)
      .filterDate('2017-01-01', '2025-12-31')
      .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
      .map(mask_s2_clouds)
      .map(add_indices))

n_images = s2.size().getInfo()
print(f"Imágenes Sentinel-2 disponibles: {n_images}")

Imágenes Sentinel-2 disponibles: 792


In [25]:
# Composite RGB 2024 para visualización
rgb_2024 = (s2.filterDate('2024-01-01', '2024-12-31')
            .select(['B4', 'B3', 'B2'])
            .median()
            .clip(aoi))

# Composite NDVI 2024
ndvi_2024 = (s2.filterDate('2024-01-01', '2024-12-31')
             .select('NDVI')
             .median()
             .clip(aoi))

Map2 = geemap.Map(center=[10.7, -74.5], zoom=10)
Map2.addLayer(rgb_2024, {'min': 0, 'max': 3000, 'bands': ['B4', 'B3', 'B2']}, 'RGB 2024')
Map2.addLayer(ndvi_2024, {'min': -0.2, 'max': 0.8, 'palette': ['red', 'yellow', 'green', 'darkgreen']}, 'NDVI 2024')
Map2.addLayer(ee.Feature(aoi), {'color': 'white'}, 'AOI', opacity=0.3)
Map2

Map(center=[10.7, -74.5], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright',…

In [26]:
# Generar composites trimestrales de NDVI, NDWI y CMRI
def quarterly_composite(year, quarter):
    month_start = (quarter - 1) * 3 + 1
    start = ee.Date.fromYMD(year, month_start, 1)
    end = start.advance(3, 'month')
    
    composite = (s2.filterDate(start, end)
                .select(['NDVI', 'NDWI', 'CMRI'])
                .median()
                .clip(aoi))
    
    n = s2.filterDate(start, end).size()
    
    return (composite
            .set('year', year)
            .set('quarter', quarter)
            .set('n_images', n)
            .set('system:time_start', start.millis()))

# Generar para 2017-2025
composites = []
for year in range(2017, 2026):
    for quarter in range(1, 5):
        composites.append(quarterly_composite(year, quarter))

quarterly_col = ee.ImageCollection(composites)
print(f"Composites trimestrales generados: {len(composites)}")

# Verificar cuántas imágenes tiene cada trimestre
for year in [2017, 2020, 2024]:
    for q in range(1, 5):
        ms = (q - 1) * 3 + 1
        start = ee.Date.fromYMD(year, ms, 1)
        end = start.advance(3, 'month')
        n = s2.filterDate(start, end).size().getInfo()
        print(f"  {year}-Q{q}: {n} imágenes")

Composites trimestrales generados: 36
  2017-Q1: 2 imágenes
  2017-Q2: 0 imágenes
  2017-Q3: 0 imágenes
  2017-Q4: 1 imágenes
  2020-Q1: 62 imágenes
  2020-Q2: 19 imágenes
  2020-Q3: 11 imágenes
  2020-Q4: 21 imágenes
  2024-Q1: 57 imágenes
  2024-Q2: 9 imágenes
  2024-Q3: 12 imágenes
  2024-Q4: 20 imágenes


In [19]:
# Ajustar: empezar desde 2018 y filtrar trimestres vacios
composites = []
for year in range(2018, 2026):
    for quarter in range(1, 5):
        composites.append(quarterly_composite(year, quarter))

quarterly_col = ee.ImageCollection(composites)
print(f"Composites trimestrales: {len(composites)} (2018-2025)")

# Exportar a Google Drive saltando trimestres sin imagenes
exportadas = 0
saltadas   = []
for year in range(2018, 2026):
    for q in range(1, 5):
        ms = (q - 1) * 3 + 1
        start = ee.Date.fromYMD(year, ms, 1)
        end   = start.advance(3, 'month')
        n = s2.filterDate(start, end).size().getInfo()
        if n == 0:
            saltadas.append(f"{year}-Q{q}")
            continue

        img = quarterly_composite(year, q)
        task = ee.batch.Export.image.toDrive(
            image=img,
            description=f'CGSM_indices_{year}_Q{q}',
            folder='CGSM_data_acotado',
            region=aoi,
            scale=10,
            crs='EPSG:4326',
            maxPixels=1e13
        )
        task.start()
        exportadas += 1

print(f"Exportaciones lanzadas: {exportadas}")
if saltadas:
    print(f"Trimestres sin imagenes (no se exportan): {', '.join(saltadas)}")
print("Monitorea: code.earthengine.google.com > Tasks")

Composites trimestrales: 32 (2018-2025)
Exportaciones lanzadas: 30
Trimestres sin imagenes (no se exportan): 2018-Q2, 2018-Q3
Monitorea: code.earthengine.google.com > Tasks


In [27]:
print(ee.data.getAssetRoots())

[{'type': 'Folder', 'id': 'projects/basic-buttress-338101/assets/AGB_CGSM_2022'}, {'type': 'Table', 'id': 'projects/basic-buttress-338101/assets/AOI_CGSM'}, {'type': 'Table', 'id': 'projects/basic-buttress-338101/assets/CGSM'}, {'type': 'Table', 'id': 'projects/basic-buttress-338101/assets/CGSM_muestras_371'}, {'type': 'Folder', 'id': 'projects/basic-buttress-338101/assets/Informe1_S2_Seca'}, {'type': 'Folder', 'id': 'projects/basic-buttress-338101/assets/Informe2_Fusion_S2_S1_4clases'}, {'type': 'Folder', 'id': 'projects/basic-buttress-338101/assets/Informe2_S2_4clases'}, {'type': 'Folder', 'id': 'projects/basic-buttress-338101/assets/Informe2_S2_Probabilities_3clases'}, {'type': 'Folder', 'id': 'projects/basic-buttress-338101/assets/Informe2_SAR_Lluviosa'}]


In [28]:
# Exportar composites RGB para SamGeo (3 períodos representativos)
periodos_rgb = [
    ('2019-01-01', '2019-06-30', 'CGSM_RGB_2019_S1'),  # Período temprano
    ('2021-01-01', '2021-06-30', 'CGSM_RGB_2021_S1'),  # Período medio
    ('2024-01-01', '2024-06-30', 'CGSM_RGB_2024_S1'),  # Período reciente
]

for start, end, name in periodos_rgb:
    rgb = (s2.filterDate(start, end)
           .select(['B4', 'B3', 'B2'])
           .median()
           .clip(aoi))
    
    # Escalar a 0-255 para SamGeo
    rgb_vis = rgb.unitScale(0, 3000).multiply(255).toByte()
    
    task = ee.batch.Export.image.toDrive(
        image=rgb_vis,
        description=name,
        folder='CGSM_data_acotado',
        region=aoi,
        scale=10,
        crs='EPSG:4326',
        maxPixels=1e13
    )
    task.start()
    print(f"Exportando: {name}")

print("3 RGBs lanzados. Los períodos definitivos se ajustarán en la Fase 2.")

Exportando: CGSM_RGB_2019_S1
Exportando: CGSM_RGB_2021_S1
Exportando: CGSM_RGB_2024_S1
3 RGBs lanzados. Los períodos definitivos se ajustarán en la Fase 2.


In [21]:
# Serie Landsat 8/9 (2013-2025)
def mask_landsat_clouds(image):
    qa = image.select('QA_PIXEL')
    cloud = qa.bitwiseAnd(1 << 3).eq(0)
    shadow = qa.bitwiseAnd(1 << 4).eq(0)
    return image.updateMask(cloud.And(shadow))

def add_landsat_indices(image):
    optical = image.select('SR_B.').multiply(0.0000275).add(-0.2)
    ndvi = optical.normalizedDifference(['SR_B5', 'SR_B4']).rename('NDVI')
    ndwi = optical.normalizedDifference(['SR_B3', 'SR_B5']).rename('NDWI')
    cmri = ndvi.subtract(ndwi).rename('CMRI')
    return image.addBands([ndvi, ndwi, cmri])

l8 = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
      .filterBounds(aoi)
      .filterDate('2013-01-01', '2025-12-31')
      .filter(ee.Filter.lt('CLOUD_COVER', 20))
      .map(mask_landsat_clouds)
      .map(add_landsat_indices))

l9 = (ee.ImageCollection('LANDSAT/LC09/C02/T1_L2')
      .filterBounds(aoi)
      .filterDate('2022-01-01', '2025-12-31')
      .filter(ee.Filter.lt('CLOUD_COVER', 20))
      .map(mask_landsat_clouds)
      .map(add_landsat_indices))

landsat = l8.merge(l9)
print(f"Imágenes Landsat 8: {l8.size().getInfo()}")
print(f"Imágenes Landsat 9: {l9.size().getInfo()}")
print(f"Total Landsat: {landsat.size().getInfo()}")

Imágenes Landsat 8: 252
Imágenes Landsat 9: 76
Total Landsat: 328


In [29]:
# Exportar composites anuales Landsat (2013-2025)
for year in range(2013, 2026):
    start = ee.Date.fromYMD(year, 1, 1)
    end = ee.Date.fromYMD(year, 12, 31)
    
    composite = (landsat.filterDate(start, end)
                .select(['NDVI', 'NDWI', 'CMRI'])
                .median()
                .clip(aoi))
    
    task = ee.batch.Export.image.toDrive(
        image=composite,
        description=f'CGSM_Landsat_indices_{year}',
        folder='CGSM_data_acotado',
        region=aoi,
        scale=30,
        crs='EPSG:4326',
        maxPixels=1e13
    )
    task.start()
    print(f"Exportando Landsat {year}")

print(f"\n{13} tareas Landsat lanzadas")

Exportando Landsat 2013
Exportando Landsat 2014
Exportando Landsat 2015
Exportando Landsat 2016
Exportando Landsat 2017
Exportando Landsat 2018
Exportando Landsat 2019
Exportando Landsat 2020
Exportando Landsat 2021
Exportando Landsat 2022
Exportando Landsat 2023
Exportando Landsat 2024
Exportando Landsat 2025

13 tareas Landsat lanzadas


In [22]:
# Guardar el polígono AOI detallado como GeoJSON
coords = aoi.coordinates().getInfo()[0]

geojson = {
    "type": "FeatureCollection",
    "features": [{
        "type": "Feature",
        "properties": {"name": "CGSM_AOI", "source": "Delimitación detallada CGSM"},
        "geometry": {"type": "Polygon", "coordinates": [coords]}
    }]
}

with open('../data/raw/cgsm_aoi.geojson', 'w') as f:
    json.dump(geojson, f, indent=2)

print("GeoJSON actualizado con polígono detallado")
print(f"Vértices: {len(coords)}")


GeoJSON actualizado con polígono detallado
Vértices: 1


In [30]:
# --- Exportar RGBs para SamGeo ---
periodos = {
    'degradacion': {'fecha_inicio': '2020-07-01', 'fecha_fin': '2020-12-31'},
    'recuperacion': {'fecha_inicio': '2022-01-01', 'fecha_fin': '2022-06-30'},
    'actual': {'fecha_inicio': '2024-07-01', 'fecha_fin': '2025-06-30'},
}

for periodo, info in periodos.items():
    rgb = (s2.filterDate(info['fecha_inicio'], info['fecha_fin'])
           .select(['B4', 'B3', 'B2'])
           .median()
           .clip(aoi))
    
    rgb_vis = rgb.unitScale(0, 3000).multiply(255).toByte()
    
    task = ee.batch.Export.image.toDrive(
        image=rgb_vis,
        description=f'CGSM_RGB_{periodo}',
        folder='CGSM_data_acotado',
        region=aoi,
        scale=10,
        crs='EPSG:4326',
        maxPixels=1e13
    )
    task.start()
    print(f"Exportando RGB {periodo}: {info['fecha_inicio']} a {info['fecha_fin']}")

print("\n3 RGBs lanzados a Google Drive (carpeta CGSM_data_acotado)")

Exportando RGB degradacion: 2020-07-01 a 2020-12-31
Exportando RGB recuperacion: 2022-01-01 a 2022-06-30
Exportando RGB actual: 2024-07-01 a 2025-06-30

3 RGBs lanzados a Google Drive (carpeta CGSM_data_acotado)


In [18]:
tasks = ee.batch.Task.list()
for task in tasks:
    desc = task.config.get('description', '')
    if 'RGB' in desc:
        print(f"  {desc}: {task.state}")

  CGSM_RGB_actual: READY
  CGSM_RGB_recuperacion: READY
  CGSM_RGB_degradacion: READY
  CGSM_RGB_2024_S1: READY
  CGSM_RGB_2021_S1: READY
  CGSM_RGB_2019_S1: READY


In [33]:
# Relanzar trimestrales 2021-Q2 hasta 2025-Q4 (los que se cancelaron por error)
trimestres_faltantes = []
for year in range(2021, 2026):
    for q in range(1, 5):
        if year == 2021 and q == 1:
            continue   # ya esta completed
        trimestres_faltantes.append((year, q))

print(f"Trimestres a relanzar: {len(trimestres_faltantes)}")

exportadas = 0
for year, q in trimestres_faltantes:
    ms = (q - 1) * 3 + 1
    start = ee.Date.fromYMD(year, ms, 1)
    end   = start.advance(3, 'month')
    n = s2.filterDate(start, end).size().getInfo()
    if n == 0:
        print(f"  {year}-Q{q}: sin imagenes, salto")
        continue

    img = quarterly_composite(year, q)
    task = ee.batch.Export.image.toDrive(
        image=img,
        description=f'CGSM_indices_{year}_Q{q}',
        folder='CGSM_data_acotado',
        region=aoi,
        scale=10,
        crs='EPSG:4326',
        maxPixels=1e13
    )
    task.start()
    exportadas += 1
    print(f"  {year}-Q{q}: lanzado ({n} imagenes)")

print(f"\nTotal lanzadas: {exportadas}")

Trimestres a relanzar: 19
  2021-Q2: lanzado (24 imagenes)
  2021-Q3: lanzado (20 imagenes)
  2021-Q4: lanzado (21 imagenes)
  2022-Q1: lanzado (43 imagenes)
  2022-Q2: lanzado (11 imagenes)
  2022-Q3: lanzado (10 imagenes)
  2022-Q4: lanzado (19 imagenes)
  2023-Q1: lanzado (54 imagenes)
  2023-Q2: lanzado (22 imagenes)
  2023-Q3: lanzado (16 imagenes)
  2023-Q4: lanzado (18 imagenes)
  2024-Q1: lanzado (57 imagenes)
  2024-Q2: lanzado (9 imagenes)
  2024-Q3: lanzado (12 imagenes)
  2024-Q4: lanzado (20 imagenes)
  2025-Q1: lanzado (39 imagenes)
  2025-Q2: lanzado (14 imagenes)
  2025-Q3: lanzado (22 imagenes)
  2025-Q4: lanzado (31 imagenes)

Total lanzadas: 19
